# HuggingFace 大模型托管网站
这个网站就是一个开源模型的托管网站，相当于模型界的 Github。

我们常见的大语言模型都是基于 Pytorch 搭建的。Pytorch 库中提供了大语言模型需要的所有组件，比如多头注意力机制、前馈神经网络、Embedding 等等，但是并没有提供整个 Decoder-only Transformer 模型。原因是市面上大部分公司并不会使用完整的 Decoder-only Transformer 模型，都会进行魔改，比如 Deepseek 会用 MoE 来取代 Feed Forward等等，因此 Pytorch 提供了所有组件来支持这些厂商来整花活。

今天用的 Transformers库也是由 HuggingFace 来维护的，其中使用 pytorch 帮我实现了大多数开源模型。当然这个库除了接入 pytorch 之外，还接入了许多其他的库，比如将模型加载到不同显卡的 Accelerate 库，还有做模型量化的 bitsandbytes 库。

# 第一步：加载已经下好的模型
从 HUggingFace 中下载我们需要的开源模型 今天要用的是DeepSeek-R1-Distill-Qwen-1.5B这个简单的模型

下载网址：https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B

下载完成后将模型保存在 model 文件夹中

In [1]:
# 加载下载好的模型
from transformers import Qwen2ForCausalLM
model = Qwen2ForCausalLM.from_pretrained("./model")
model

/opt/miniconda3/envs/torch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████████████████| 339/339 [00:00<00:00, 7640.68it/s]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [2]:
# 我们看一下加载的model是不是pytorch中的Module类型
from torch.nn import Module
isinstance(model, Module)

True

In [5]:
#其他模型加载方法
# 除了上述加载模型的方式外还可以使用 AutoModelForCausalLM 来统一加载所有对话形式的大语言模型
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained("./model")
#还有一种自动下载并加载模型的方法
#比如使用AutoModelForCausalLM.from_pretrained("DeepSeek-R1-Distill-Qwen-1.5B")，就可以直接从网上下载并加载模型
model

Loading weights: 100%|█████████████████████| 339/339 [00:00<00:00, 13378.14it/s]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

# 第二步：模型加载完成后，要考虑使用硬件加速
模型加载到内存中后，只有 CPU 才能访问内存

如果要让 GPU，或者 NPU 加速，需要将模型加载到显存中

In [6]:
model.to('mps') #mac用 mps，window 用 cuda
model.device

device(type='mps', index=0)

# 第三步：加载分词器
利用 tokenizer 将输入文字处理成 AI 大模型能处理的 token。

分词器的常见算法主要分为三种：BPE、WordPiece 和 Unigram。

TokenizersBackend 是 transformers 这个库中可以支持所有算法的分词器。

In [7]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("./model")
tokenizer

TokenizersBackend(name_or_path='./model', vocab_size=151643, model_max_length=16384, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<｜begin▁of▁sentence｜>', 'eos_token': '<｜end▁of▁sentence｜>', 'pad_token': '<｜end▁of▁sentence｜>'}, added_tokens_decoder={
	151643: AddedToken("<｜end▁of▁sentence｜>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<｜User｜>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	151645: AddedToken("<｜Assistant｜>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	151646: AddedToken("<｜begin▁of▁sentence｜>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151647: AddedToken("<|EOT|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	151648: AddedToken("<think>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	151649: AddedT

In [8]:
# 将一个输入信息转化为 token，并传入显存
inputs = tokenizer(
    [
        "hello，tell me your name please.",
        "10*5="
    ],
    padding=True,    # 模型可以好几句话同时接龙，但是模型内是矩阵乘法，需要保证输入句子长度相同，因此需要填充
    padding_side="left", # 由于模型是在输入句子的结尾进行接龙，因此填充需要填在最左边
    return_tensors="pt"  # tokenizer返回的是python 数组，pytorch 推理需要传入 tensor，所以这个做了一个转换
).to(model.device)

inputs

{'input_ids': tensor([[ 14990,   3837,  72357,    752,    697,    829,   4486,     13],
        [151643, 151643, 151643,     16,     15,      9,     20,     28]],
       device='mps:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 1, 1, 1, 1, 1]], device='mps:0')}

In [9]:
tokenizer.batch_decode(inputs.input_ids) # 将句子翻译出来看一下填充的内容

['hello，tell me your name please.',
 '<｜end▁of▁sentence｜><｜end▁of▁sentence｜><｜end▁of▁sentence｜>10*5=']

# 第三步：利用 LLM 进行推理
我们已经将输入内容处理成大语言模型能够处理的 token，下一步就是进行推理，完成词语接龙。

In [10]:
outputs = model.generate(
    input_ids = inputs.input_ids,
    attention_mask=inputs.attention_mask, # 反应哪些是填充的内容，哪些是用户输入的内容
    max_new_tokens = 32  #最多输出多少个 token

)
outputs

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


tensor([[ 14990,   3837,  72357,    752,    697,    829,   4486,     13,    358,
           1184,    697,    829,     13,    358,    646,    944,   3291,    498,
            847,    829,   1576,    498,   1513,    944,   1366,    752,    311,
           1414,     13,   1988,    498,   1521,   1977,    697,    829,   3055,
             11,    773,    358,    646],
        [151643, 151643, 151643,     16,     15,      9,     20,     28,     20,
             15,    271,   4416,     11,    220,     20,     15,     14,     20,
             15,     28,     16,    271,  80022,     11,    773,    429,   1035,
           3076,    279,  11341,    374,    220,     16,     25,     16,     11,
            714,    358,   1414,    429]], device='mps:0')

In [11]:
# 将结果解码看一下是什么内容：
tokenizer.batch_decode(outputs)

["hello，tell me your name please. I need your name. I can't tell you my name because you don't want me to know. But you did say your name once, so I can",
 '<｜end▁of▁sentence｜><｜end▁of▁sentence｜><｜end▁of▁sentence｜>10*5=50\n\nSo, 50/50=1\n\nHmm, so that would mean the ratio is 1:1, but I know that']

# 第四步：分析模型胡说八道的原因并进行修正
模型一般是使用大量文章来对模型进行预训练，也就是文字接龙，但是实际使用中，还需要对模型进行微调才能实现对话。

在模型微调中，一般都会有相对固定的格式，比如：

user：你好

Assistant：你也好

或者有的模型还会在模型中加入思考和推理的过程。

因此我们需要对输入的文本进行一个格式的转化，让模型能更好的识别。

In [12]:
# 修改输入内容的格式
message = [
    [
        {"role":"user", "content": "hello，tell me your name please."}
    ],
    [
        {"role":"user", "content": "10*5="}
    ]
]

tokenizer.padding_side = "left"

# tokenizer中封装了一个chat_template脚本，负责将 message 这种格式转化成模型训练时用到的对话格式
inputs = tokenizer.apply_chat_template(
    message,
    return_tensors="pt",
    padding=True,
    add_generation_prompt=True    # 生成的token会跟上Assistant的标签和思维链的起始标志
).to(model.device)

tokenizer.batch_decode(inputs.input_ids)

['<｜begin▁of▁sentence｜><｜User｜>hello，tell me your name please.<｜Assistant｜><think>\n',
 '<｜end▁of▁sentence｜><｜end▁of▁sentence｜><｜end▁of▁sentence｜><｜begin▁of▁sentence｜><｜User｜>10*5=<｜Assistant｜><think>\n']

In [13]:
outputs = model.generate(
    input_ids = inputs.input_ids,
    attention_mask=inputs.attention_mask, # 反应哪些是填充的内容，哪些是用户输入的内容
    max_new_tokens = 256,  #最多输出多少个 token
    pad_token_id=tokenizer.eos_token_id,  # 消除警告，告诉它该结束了
    do_sample=True,  # 开启采样，让对话更自然
)
# 将结果解码看一下是什么内容：
tokenizer.batch_decode(outputs, skip_special_tokens=True)

["<｜User｜>hello，tell me your name please.<｜Assistant｜><think>\nGreetings! I'm DeepSeek-R1, an artificial intelligence assistant created by DeepSeek. I'm at your service and would be delighted to assist you with any inquiries or tasks you may have.\n</think>\n\nGreetings! I'm DeepSeek-R1, an artificial intelligence assistant created by DeepSeek. I'm at your service and would be delighted to assist you with any inquiries or tasks you may have.",
 "<｜User｜>10*5=<｜Assistant｜><think>\nTo solve the multiplication problem 10 multiplied by 5, I'll start by recognizing that multiplying by 10 is equivalent to adding a zero at the end of the number.\n\nTherefore, 10 multiplied by 5 is simply 50.\n</think>\n\n**Solution:**\n\nTo solve the multiplication problem \\(10 \\times 5\\), follow these easy steps:\n\n1. **Understand the Multiplication:**\n   \n   Multiplying 10 by 5 means adding 10 five times:\n   \\[\n   10 + 10 + 10 + 10 + 10 = 50\n   \\]\n\n2. **Alternatively, Use Place Value:**\n   \n 

# 一些优化
我们发现上面的输出基本是对的，但是不够优优雅。

包含了很多不需要的符号。

并且每次都需要手动处理输入文本，不能连续对话。

因此，我们需要进行两方面的优化：

第一：过滤掉一些不需要的符号。

第二：写一个 for 循环可以实现连续对话

In [15]:
messages = []

print("="*50)
print("欢迎与 DeepSeek-R1 交流！输入 'quit' 或 'exit' 结束对话。")
print("="*50)


# 4. 开启连续对话循环
while True:
    # 获取用户在终端的输入
    user_input = input("\nUser: ")

    # 设置退出指令
    if user_input.lower() in ['quit', 'exit']:  # 将字符串中所有大写字母转换为小写字母
        print("结束对话，再见！")
        break
    if not user_input.strip(): # 去除两端空白字符
        continue

    # A. 将用户的最新发言追加到历史记录中
    messages.append({"role": "user", "content": user_input})

    # B. 应用 Chat Template 处理包含历史记录的完整对话
    # 每次传入的都是不断增长的 messages 列表
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,  # 自动加上 <|Assistant|> 的起始标志
        return_tensors="pt"
    ).to("mps")

    # 记录当前输入 Token 的长度
    input_length = inputs.input_ids.shape[1]

    # C. 模型推理生成回复
    outputs = model.generate(
        inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=512,  # 控制每次最大输出长度
        pad_token_id=tokenizer.eos_token_id,  # 消除警告，告诉它遇到结束符该结束了
        do_sample=True,  # 开启采样，让对话更自然
        temperature=0.7  # 控制回答的创造性和发散度
    )

    # D. 核心切片：将包含输入和输出的完整 Tensor 进行切片，只保留本次新生成的部分
    new_tokens = outputs[0][input_length:]

    # 解码生成的 Token 变为可读文本
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # 如果输出中包含 </think>，我们就以它为界限进行分割，只取最后面的那部分（即真正的回答）
    if "</think>" in response:
        clean_response = response.split("</think>")[-1].strip()
    else:
        clean_response = response.strip()
    # -----------------------

    print(f"\nAssistant: {clean_response}")

    # E. 将模型的回复加入历史记录，完成这一轮的上下文闭环
    messages.append({"role": "assistant", "content": response})

欢迎与 DeepSeek-R1 交流！输入 'quit' 或 'exit' 结束对话。

User: 你好，你叫什么名字

Assistant: 您好！我是由中国的深度求索（DeepSeek）公司开发的智能助手DeepSeek-R1。如您有任何任何问题，我会尽我所能为您提供帮助。

User: 你可以做什么？

Assistant: 您好！我是由中国的深度求索（DeepSeek）公司开发的智能助手DeepSeek-R1。您问的问题非常广泛，我可以为您提供以下内容：

1. **数据分析**：帮助您处理、分析和可视化数据，提供见解和建议。
2. **模型开发**：协助您开发和部署AI模型，解决复杂问题。
3. **市场调研**：帮助您进行市场分析和预测，为商业决策提供数据支持。
4. **算法优化**：提供算法优化建议，提升AI系统性能。
5. **文档编写**：帮助您编写技术文档和报告。
6. **内容生成**：提供创意内容，如文章、视频等。
7. **数据分析工具**：推荐和使用各种数据分析工具，提升效率。
8. **技术咨询**：针对具体的技术问题，提供解决方案。

如果您有具体的请求或需求，请告诉我，我会尽力为您提供帮助！

User: quit
结束对话，再见！
